In [1]:
import os
import re
import pandas as pd

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# === DETECTION KEYWORDS ===
REAL_DEVICE_KEYWORDS = [
    'adb devices', 'adb get-state', 'adb get-serialno', 'adb install', 'adb install -r',
    'adb -s', 'adb shell', 'adb root', 'adb shell settings', 'adb shell input', 'adb shell pm grant'
]

EMULATOR_KEYWORDS = [
    'emulator',                   # Only if used to launch (not just mentioned)
    'android-wait-for-emulator', # Utility script for emulator wait
    'start-emulator.sh',         # Custom wrapper
    'avdmanager create avd',     # Explicit AVD setup
    'emulator -avd',             # Launch emulator
    'emulator @',                # Launch by name
]


THIRD_PARTY_KEYWORDS = [
    'gcloud firebase test android run', 'browserstack', 'saucectl', 'bstack', 'appcenter test run',
    'test_matrix.json', 'firebase.json'
]

INSTRUMENTATION_TRIGGER_KEYWORDS = [
    'adb shell am instrument', 'am instrument', './gradlew connectedandroidtest',
    'connectedcheck', 'connectedflavortest', 'createinstrumentationtestcoveragereport',
    'runinstrumentationtests', 'executescreenshottests', 'orchestrator', 'connectedtest'
]

UNIT_TEST_KEYWORDS_CI = [
    'gradlew test', './gradlew test', './gradlew jvmtest',
    'testdebugunittest', 'testreleaseunittest', 'kotlintest',
    'unittest', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

UNIT_TEST_KEYWORDS_BUILD = ['junit', 'testimplementation']
INSTRUMENT_TEST_KEYWORDS_BUILD = ['androidtestimplementation', 'espresso', 'uiautomator']

STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

# === MAIN ANALYSIS ===
results = []

for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue

    file_ext = os.path.splitext(file)[-1].lower()
    if file_ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml']:
        continue

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except:
            pass

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().lower()

        parsed_ok = True
        file_is_build = file_ext in ['.gradle', '.kts']

        # === INIT VALUES ===
        device_setup = set()
        trigger_detected = False
        test_definition = False
        unit_test_ci = False
        unit_test_build = False
        instru_test_status = "None"

        # === BUILD FILES: Only Analyze Test Definition & Build Unit Test ===
        if file_is_build:
            if any(k in content for k in UNIT_TEST_KEYWORDS_BUILD):
                unit_test_build = True
            if any(k in content for k in INSTRUMENT_TEST_KEYWORDS_BUILD):
                test_definition = True

        else:
            # === DEVICE SETUP ===
            if any(k in content for k in REAL_DEVICE_KEYWORDS):
                device_setup.add("Real_Device")
            if any(k in content for k in EMULATOR_KEYWORDS):
                device_setup.add("Emulator")
            if any(k in content for k in THIRD_PARTY_KEYWORDS):
                device_setup.add("Third_Party_Lab")

            # === TEST TRIGGER ===
            if any(k in content for k in INSTRUMENTATION_TRIGGER_KEYWORDS):
                trigger_detected = True

            # === UNIT TEST (CI) ===
            if any(k in content for k in UNIT_TEST_KEYWORDS_CI):
                unit_test_ci = True

            # === STATUS LOGIC ===
            platform_key = ci_platform.strip().lower()
            ds = "None" if not device_setup else ", ".join(sorted(device_setup))
            ht = "Yes" if trigger_detected else "No"

            if file_ext in ['.yml', '.yaml']:
                if ds == "None" and ht == "Yes":
                    if platform_key in STRICT_CI_PLATFORMS:
                        instru_test_status = "Defective"
                    elif platform_key in LENIENT_CI_PLATFORMS:
                        instru_test_status = "Complete"
                elif ds != "None" and ht == "Yes":
                    instru_test_status = "Complete"
                elif ds != "None" and ht == "No":
                    instru_test_status = "Manual"


    except Exception as e:
        parsed_ok = False
        ds = "None"
        ht = "No"
        instru_test_status = "Error while parsing"

    # === FINALIZE FIELDS FOR EXPORT ===
    results.append({
        'filename': file,
        'file_type': file_ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup': "None" if file_is_build else (", ".join(sorted(device_setup)) if device_setup else "None"),
        'has_trigger': "No" if file_is_build else ("Yes" if trigger_detected else "No"),
        'Test_Definition': "Yes" if test_definition else "No",
        'unit_test_ci': False if file_is_build else unit_test_ci,
        'unit_test_build': unit_test_build,
        'instru_test_status': instru_test_status,
        'parsed_ok': parsed_ok
    })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete! Results saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete! Results saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv
